# 推荐系统 A/B 测试效果验证模拟 (多场景分析)

本 Notebook 旨在探讨在不同的“样本量”与“算法提升幅度”下，A/B 测试统计显著性的变化。这非常考验产品经理对 **样本量计算 (Sample Size)** 和 **最小可检测效应 (MDE)** 的深刻理解。


In [15]:
import numpy as np
from scipy import stats
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

def run_ab_simulation(N_control, N_treatment, ctr_c, ctr_t, cvr_c, cvr_t, aov_mean_c, aov_mean_t):
    np.random.seed(42)

    clicks_c = np.random.binomial(1, ctr_c, N_control)
    clicks_t = np.random.binomial(1, ctr_t, N_treatment)
    t_clicks_c, t_clicks_t = clicks_c.sum(), clicks_t.sum()

    orders_c = np.random.binomial(1, cvr_c, t_clicks_c)
    orders_t = np.random.binomial(1, cvr_t, t_clicks_t)
    t_orders_c, t_orders_t = orders_c.sum(), orders_t.sum()

    aov_c = np.random.lognormal(aov_mean_c, 0.4, t_orders_c)
    aov_t = np.random.lognormal(aov_mean_t, 0.4, t_orders_t)

    ctr_obs_c, ctr_obs_t = t_clicks_c / N_control, t_clicks_t / N_treatment
    cvr_obs_c, cvr_obs_t = t_orders_c / t_clicks_c, t_orders_t / t_clicks_t
    aov_obs_c, aov_obs_t = aov_c.mean() if t_orders_c > 0 else 0, aov_t.mean() if t_orders_t > 0 else 0

    chi2_ctr, p_val_ctr, _, _ = stats.chi2_contingency([[t_clicks_c, N_control - t_clicks_c], [t_clicks_t, N_treatment - t_clicks_t]])
    chi2_cvr, p_val_cvr, _, _ = stats.chi2_contingency([[t_orders_c, t_clicks_c - t_orders_c], [t_orders_t, t_clicks_t - t_orders_t]])
    t_stat, p_val_aov = stats.ttest_ind(aov_c, aov_t, equal_var=False) if t_orders_c > 1 and t_orders_t > 1 else (0, 1) 

    def print_metric(name, v1, v2, p, is_curr=False):
        sig = '显著' if p < 0.05 else '不显著'
        fmt = lambda x: f"￥{x:.2f}" if is_curr else f"{x*100:.2f}%"
        print(f"[{name}] Control: {fmt(v1):<8} | Treatment: {fmt(v2):<8} | P-value: {p:.4f} -> {sig}")
        
    print_metric("点击率 (CTR)", ctr_obs_c, ctr_obs_t, p_val_ctr)
    print_metric("转化率 (CVR)", cvr_obs_c, cvr_obs_t, p_val_cvr)
    print_metric("客单价 (AOV)", aov_obs_c, aov_obs_t, p_val_aov, is_curr=True)


### 场景一：基线原始状况（初始状态）
- **实验设定**：样本量小（各 5 万曝光）。算法带来微小但是真实的转化率提升（真实 CVR：`10.0% -> 10.8%`）。
- **产研分析**：由于最终走到支付层级的用户只有两三百人，这几单的微弱差距无法在严谨的统计学上证明不是“偶然的运气”。这说明当前流量池不足以验证这种小幅度优化。


In [16]:
print("========== 场景一：小流量 + 小幅度提升 ==========")
run_ab_simulation(
    N_control=50000, N_treatment=50000,
    ctr_c=0.050, ctr_t=0.055,
    cvr_c=0.100, cvr_t=0.108,
    aov_mean_c=np.log(100), aov_mean_t=np.log(108)
)


### 场景二：大盘流量加持（靠时间验证）
- **实验设定**：样本量翻倍暴增（各 50 万曝光）。算法本身没有任何进步，依然是那个微小的转化率提升（真实 CVR 维持 `10.0% -> 10.8%` 性能）。
- **产研分析**：在数十万大军的样本下（数千笔累积订单），极其微弱的优势也足以被系统捕捉并确认这“绝非偶然”。这是大多数大厂迭代优化的基调：**算准 MDE，靠时间与大流量换取对结果的确信度**。


In [17]:
print("\n========== 场景二：大流量 + 小幅度提升 ==========")
run_ab_simulation(
    N_control=500000, N_treatment=500000,
    ctr_c=0.050, ctr_t=0.055,
    cvr_c=0.100, cvr_t=0.108,
    aov_mean_c=np.log(100), aov_mean_t=np.log(108)
)



========== 场景二：大流量 + 小幅度提升 ==========
[点击率 (CTR)] Control: 4.99%    | Treatment: 5.57%    | P-value: 0.0000 -> ✅ 显著
[转化率 (CVR)] Control: 9.98%    | Treatment: 10.96%   | P-value: 0.0003 -> ✅ 显著
[客单价 (AOV)] Control: ￥109.26  | Treatment: ￥115.74  | P-value: 0.0000 -> ✅ 显著


### 场景三：算法产生质变（核心技术碾压）
- **实验设定**：回到极恶劣的小样本环境（各 5 万曝光）。但是此次算法上线了极其强悍的 AI 模型，完美击中用户痛点，让真实转化率飙升（真实 CVR：`10.0% -> 14.0%`）。
- **产研分析**：**这是碾压级别的降维打击。** 即使冷启动流量少得可怜，但实验组用户转化意愿存在质的飞跃。这无需经年累月跑流量，短期内就能靠显著的结果拿到 100% 全量发布的通行证。


In [18]:
print("\n========== 场景三：小流量 + 大幅度提升 ==========")
run_ab_simulation(
    N_control=50000, N_treatment=50000,
    ctr_c=0.050, ctr_t=0.060,
    cvr_c=0.100, cvr_t=0.140,
    aov_mean_c=np.log(100), aov_mean_t=np.log(150)
)



========== 场景三：小流量 + 大幅度提升 ==========
[点击率 (CTR)] Control: 4.94%    | Treatment: 5.79%    | P-value: 0.0000 -> ✅ 显著
[转化率 (CVR)] Control: 10.16%   | Treatment: 14.12%   | P-value: 0.0000 -> ✅ 显著
[客单价 (AOV)] Control: ￥108.39  | Treatment: ￥166.84  | P-value: 0.0000 -> ✅ 显著
